# 🚀 Auto Re-up TikTok / Douyin — GPU Worker Node (T4 Free)
### Mô hình Lai (Hybrid Architecture) — Tận dụng tối đa GPU NVIDIA Tesla T4 16GB VRAM

Notebook này biến Google Colab thành một **GPU AI Microservice** chuyên xử lý các tác vụ cực nặng:
- 🎙️ **Faster-Whisper (CUDA FP16):** Bóc tách phụ đề tự động siêu tốc.
- 🗣️ **VieNeu-TTS:** Lồng tiếng giọng đọc AI tiếng Việt chất lượng cao.
- 🎵 **Audio-Separator (UVR5 / Demucs):** Tách giọng nói và nhạc nền giữ trọn hiệu ứng âm thanh.
- 🎬 **FFmpeg Hardware Render (NVENC):** Xuất video lách bản quyền với tốc độ 60fps+.

---

### 📌 Bước 1: Kiểm Tra Card Đồ Họa NVIDIA Tesla T4
*Hãy chắc chắn bạn đã chọn môi trường: **Runtime -> Change runtime type -> T4 GPU**.*

In [ ]:
!nvidia-smi

import torch
print("===================================================")
print(f"CUDA Available:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM:               {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("===================================================")

### 📁 Bước 2: Kết Nối Google Drive (Lưu Trữ Dữ Liệu Vĩnh Viễn)
*Giúp lưu video thành phẩm, model weights cache và file cấu hình, không lo bị mất khi Colab tắt.*

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

STORAGE_DIR = "/content/drive/MyDrive/auto_reup_storage"
os.makedirs(os.path.join(STORAGE_DIR, "outputs"), exist_ok=True)
os.makedirs(os.path.join(STORAGE_DIR, "uploads"), exist_ok=True)
print(f"✅ Đã thiết lập thư mục lưu trữ vĩnh viễn tại: {STORAGE_DIR}")

### ⚙️ Bước 3: Cài Đặt Các Gói Hệ Thống & Thư Viện AI
*Cài FFmpeg, eSpeak NG (phục vụ Vieneu), Cloudflared và các thư viện Python tối ưu cho GPU T4.*

In [ ]:
# 1. Cài đặt các gói hệ thống Linux
!apt-get update -qq
!apt-get install -y -qq ffmpeg espeak-ng wget curl

# 2. Tải và cài đặt Cloudflared để mở Tunnel HTTPS miễn phí
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 || apt-get install -f -y > /dev/null 2>&1

# 3. Cài đặt các thư viện AI chuyên dụng
!pip install -q fastapi uvicorn python-multipart pydantic httpx aiofiles
!pip install -q faster-whisper
!pip install -q vieneu
!pip install -q audio-separator onnxruntime-gpu pydub pysrt
!pip install -q imageio-ffmpeg opencv-python-headless Pillow

print("✅ Cài đặt môi trường AI hoàn tất!")

### 🚀 Bước 4: Khởi Chạy GPU Worker Server & Mở Đường Hầm Cloudflare
*Sau khi chạy cell này, một đường dẫn **HTTPS công khai** (`https://xxx.trycloudflare.com`) sẽ xuất hiện. Hãy copy đường dẫn đó về máy tính cá nhân.*

In [ ]:
import subprocess
import time
import re
import os

# Dừng các tiến trình cũ nếu có
!pkill -f "uvicorn colab_server:app" || true
!pkill -f "cloudflared tunnel" || true
time.sleep(1)

# Khởi động FastAPI GPU Server trong nền
print("[*] Đang khởi chạy FastAPI GPU Server...")
!nohup python3 -m uvicorn colab_server:app --host 0.0.0.0 --port 8000 > /content/gpu_server.log 2>&1 &

# Chờ server khởi động thành công
time.sleep(3)

# Khởi động Cloudflare Tunnel
print("[*] Đang kết nối Cloudflare Tunnel...")
!rm -f /content/tunnel.log
!nohup cloudflared tunnel --url http://127.0.0.1:8000 > /content/tunnel.log 2>&1 &

# Tìm URL HTTPS từ log
tunnel_url = None
for i in range(25):
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break
    time.sleep(1)

print("\n" + "="*60)
if tunnel_url:
    print("🎉 CHÚC MỪNG! GPU WORKER CỦA BẠN ĐÃ ONLINE!")
    print("\n👉 ĐỊA CHỈ GPU CLOUD CỦA BẠN:")
    print(f"   {tunnel_url}")
    print("\n👉 KIỂM TRA TRẠNG THÁI (Swagger API Docs):")
    print(f"   {tunnel_url}/docs")
    print("="*60)
    print("\nHãy copy địa chỉ trên để kết nối từ máy tính cá nhân của bạn!")
else:
    print("⚠️ Đang khởi tạo Tunnel, kiểm tra log: !cat /content/tunnel.log")

### 🛡️ Bước 5: Giữ Phiên Colab Luôn Hoạt Động (Anti-Disconnect)
Chạy cell dưới đây để hiển thị console log thời gian thực hoặc mở F12 trên trình duyệt và dán đoạn mã JavaScript sau vào Tab Console để tránh bị Colab tự ngắt kết nối:

```javascript
function ClickConnect(){
    console.log("Keep-alive tick: " + new Date().toLocaleTimeString());
    document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click();
}
setInterval(ClickConnect, 60000);
```

In [ ]:
# Theo dõi log GPU Server theo thời gian thực
!tail -f /content/gpu_server.log